# Geometric Deep Learning RAG

We creat a simple RAG system using a HuggingFace model based on Geometric Deep Learning: https://geometricdeeplearning.com

In [3]:
!pip install pypdf langchain langchain-community sentence-transformers transformers torch faiss-cpu ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 2.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 2.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 2.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 MB 2.0 MB/s eta 0:00:0000:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 2.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 2.0 MB/s eta 0:00:00-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 2.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 2.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 2.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 2.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 640.3/640.3 kB 2.0 MB/s eta 0:00:00-:--:--
  Attempting uninstall: zstandard
    Found existing in

In [83]:
# Advanced Free PDF RAG System - Optimized for Technical Documents
# Install required packages first:
# !pip install pypdf langchain langchain-community sentence-transformers transformers torch faiss-cpu

import os
from typing import List, Dict, Any
import numpy as np
import re

# Core libraries
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.schema import Document
from transformers import pipeline

class TechnicalPDFRag:
    def __init__(self):
        """
        Initialize RAG system optimized for technical documents
        """
        print("🚀 Initializing Technical PDF RAG System...")
        
        # Use better embedding model for technical content
        print("Loading embeddings model (optimized for technical content)...")
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-mpnet-base-v2",  # Better for technical content
            model_kwargs={'device': 'cpu'}
        )
        
        # Use a more capable text generation model
        print("Loading text generation model...")
        self.text_generator = pipeline(
            "text-generation",
            model="microsoft/DialoGPT-medium",  # Better for coherent responses
            max_length=512,
            do_sample=True,
            temperature=0.3,  # Lower temperature for more focused responses
            pad_token_id=50256
        )
        
        self.vectorstore = None
        self.documents = []
        self.pdf_name = None
        print("✅ Technical RAG system ready!")
    
    def load_pdf_from_path(self, pdf_path: str) -> List[Document]:
        """Load PDF with optimized chunking for technical content"""
        if not os.path.exists(pdf_path):
            print(f"❌ PDF file not found: {pdf_path}")
            pdf_files = [f for f in os.listdir('.') if f.endswith('.pdf')]
            if pdf_files:
                print("📄 Available PDF files:")
                for f in pdf_files:
                    print(f"   • {f}")
            raise FileNotFoundError(f"PDF file not found: {pdf_path}")
        
        print(f"📄 Loading PDF: {pdf_path}")
        self.pdf_name = pdf_path
        
        # Load PDF
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
        
        # Optimized text splitter for technical content
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1500,  # Larger chunks for technical context
            chunk_overlap=300,  # More overlap to maintain mathematical concepts
            length_function=len,
            separators=["\n\n", "\n", ". ", "? ", "! ", " ", ""]  # Better splitting
        )
        
        split_docs = text_splitter.split_documents(documents)
        self.documents = split_docs
        
        print(f"✅ Loaded {len(documents)} pages, split into {len(split_docs)} chunks")
        return split_docs
    
    def create_vectorstore(self):
        """Create vectorstore optimized for technical content"""
        if not self.documents:
            raise ValueError("No documents loaded. Call load_pdf_from_path() first.")
        
        print("🔄 Creating embeddings optimized for technical content...")
        print("⏳ This may take a few minutes for technical documents...")
        
        self.vectorstore = FAISS.from_documents(self.documents, self.embeddings)
        print("✅ Technical vectorstore created!")
    
    def query(self, question: str, k: int = 4) -> Dict[str, Any]:
        """
        Smart query system that creates coherent answers for technical questions
        """
        if not self.vectorstore:
            raise ValueError("Vectorstore not created. Call create_vectorstore() first.")
        
        # Get relevant documents
        docs = self.vectorstore.similarity_search(question, k=k)
        
        # Create clean, comprehensive context
        context_parts = []
        for doc in docs:
            # Clean and prepare text
            text = doc.page_content.replace('\n', ' ').strip()
            # Remove excessive whitespace
            text = re.sub(r'\s+', ' ', text)
            if len(text) > 50:  # Only include substantial content
                context_parts.append(text)
        
        # Create response using the most relevant context
        answer = self._generate_technical_answer(question, context_parts)
        
        return {
            "question": question,
            "answer": answer,
            "source_documents": docs,
            "method": "technical_synthesis"
        }
    
    def _generate_technical_answer(self, question: str, context_parts: List[str]) -> str:
        """Generate coherent answer for technical questions"""
        if not context_parts:
            return "I couldn't find relevant information in the document to answer that question."
        
        # Strategy 1: Look for direct explanations
        direct_answer = self._find_direct_explanation(question, context_parts)
        if direct_answer:
            return direct_answer
        
        # Strategy 2: Synthesize from multiple contexts
        synthesis = self._synthesize_context(question, context_parts[:3])
        if synthesis:
            return synthesis
        
        # Strategy 3: Return most relevant context with explanation
        best_context = context_parts[0]
        return f"Based on the document: {best_context[:400]}{'...' if len(best_context) > 400 else ''}"
    
    def _find_direct_explanation(self, question: str, context_parts: List[str]) -> str:
        """Look for direct explanations in the text"""
        question_keywords = set(question.lower().split())
        
        for context in context_parts:
            sentences = context.split('. ')
            
            for i, sentence in enumerate(sentences):
                sentence_words = set(sentence.lower().split())
                
                # Check for high keyword overlap
                overlap = len(question_keywords.intersection(sentence_words))
                
                if overlap >= 2 and len(sentence) > 50:
                    # Found a relevant sentence, try to get surrounding context
                    start_idx = max(0, i-1)
                    end_idx = min(len(sentences), i+3)
                    
                    explanation = '. '.join(sentences[start_idx:end_idx])
                    
                    # Clean up the explanation
                    explanation = explanation.strip()
                    if not explanation.endswith('.'):
                        explanation += '.'
                    
                    if len(explanation) > 100:  # Ensure substantial answer
                        return explanation
        
        return None
    
    def _synthesize_context(self, question: str, context_parts: List[str]) -> str:
        """Synthesize information from multiple contexts"""
        # Combine relevant parts
        combined_context = " ".join(context_parts[:2])  # Use top 2 contexts
        
        # Look for key phrases that might answer the question
        question_lower = question.lower()
        
        # Common technical question patterns
        if "why" in question_lower or "how" in question_lower:
            # Look for explanatory text
            explanatory_phrases = [
                "because", "since", "due to", "therefore", "thus", "hence",
                "this is", "this means", "the reason", "explanation", "mechanism"
            ]
            
            sentences = combined_context.split('. ')
            relevant_sentences = []
            
            for sentence in sentences:
                if (any(phrase in sentence.lower() for phrase in explanatory_phrases) and 
                    len(sentence) > 30):
                    relevant_sentences.append(sentence)
            
            if relevant_sentences:
                return '. '.join(relevant_sentences[:2]) + '.'
        
        # For definition questions
        if "what is" in question_lower or "define" in question_lower:
            # Look for definitions
            definition_phrases = ["is defined as", "refers to", "means", "is a", "are"]
            
            sentences = combined_context.split('. ')
            for sentence in sentences:
                if (any(phrase in sentence.lower() for phrase in definition_phrases) and 
                    len(sentence) > 30):
                    return sentence + ('.' if not sentence.endswith('.') else '')
        
        return None
    
    def similarity_search(self, query: str, k: int = 5):
        """Find most similar document chunks"""
        if not self.vectorstore:
            raise ValueError("Vectorstore not created")
        return self.vectorstore.similarity_search(query, k=k)
    
    def save_vectorstore(self, path: str = "technical_vectorstore"):
        """Save vectorstore to disk"""
        if not self.vectorstore:
            raise ValueError("No vectorstore to save")
        self.vectorstore.save_local(path)
        print(f"💾 Vectorstore saved to {path}")
    
    def load_vectorstore(self, path: str = "technical_vectorstore"):
        """Load vectorstore from disk"""
        if not os.path.exists(path):
            raise FileNotFoundError(f"Vectorstore not found at {path}")
        self.vectorstore = FAISS.load_local(path, self.embeddings, allow_dangerous_deserialization=True)
        print(f"📂 Vectorstore loaded from {path}")

# Improved helper functions
def setup_technical_rag():
    """Setup RAG system optimized for technical documents"""
    print("🧠 TECHNICAL PDF RAG SETUP")
    print("=" * 40)
    print("Optimized for: Math, CS, Physics, Engineering docs")
    print()
    
    rag = TechnicalPDFRag()
    
    print("📋 NEXT STEPS:")
    print("1. Upload PDF to Jupyter (drag & drop or Upload button)")
    print("2. Run: list_pdf_files()")
    print("3. Run: rag.load_pdf_from_path('filename.pdf')")
    print("4. Run: rag.create_vectorstore()")
    print("5. Ask: ask_question(rag, 'your question')")
    
    return rag

def ask_question(rag_system, question: str, show_sources: bool = False):
    """Ask technical questions and get coherent answers"""
    try:
        result = rag_system.query(question)
        
        # Print clean answer
        answer = result.get('answer', 'No answer found')
        print(answer)
        
        # Show sources if requested
        if show_sources:
            docs = result.get('source_documents', [])
            if docs:
                pages = [str(doc.metadata.get('page', '?')) for doc in docs]
                unique_pages = sorted(list(set(pages)))
                print(f"\n(Sources: pages {', '.join(unique_pages)})")
        
        # Don't return anything to avoid double display
        
    except Exception as e:
        print(f"Error: {e}")

def ask_with_sources(rag_system, question: str):
    """Ask question and show page sources"""
    ask_question(rag_system, question, show_sources=True)

def search_document(rag_system, query: str, num_results: int = 3):
    """Search for specific content in the document"""
    try:
        docs = rag_system.similarity_search(query, k=num_results)
        
        print(f"🔍 Search results for: '{query}'")
        print("=" * 50)
        
        for i, doc in enumerate(docs, 1):
            page = doc.metadata.get('page', '?')
            content = doc.page_content.replace('\n', ' ')[:200]
            print(f"{i}. Page {page}: {content}...")
            print()
        
        return docs
    except Exception as e:
        print(f"Search error: {e}")
        return []

def list_pdf_files():
    """List available PDF files"""
    pdf_files = [f for f in os.listdir('.') if f.endswith('.pdf')]
    if pdf_files:
        print("📄 Available PDF files:")
        for i, file in enumerate(pdf_files, 1):
            print(f"   {i}. {file}")
    else:
        print("❌ No PDF files found")
        print("💡 Upload a PDF using Jupyter's file browser")
    return pdf_files

# Google Colab support
def colab_setup():
    """Easy setup for Google Colab"""
    try:
        from google.colab import files
        print("📤 Upload your PDF file:")
        uploaded = files.upload()
        
        if uploaded:
            filename = list(uploaded.keys())[0]
            print(f"✅ Uploaded: {filename}")
            
            rag = TechnicalPDFRag()
            rag.load_pdf_from_path(filename)
            rag.create_vectorstore()
            
            print("🎉 Technical RAG ready for complex questions!")
            return rag
        
    except ImportError:
        print("❌ Not in Google Colab. Use setup_technical_rag() instead.")
    return None

print("🧠 TECHNICAL PDF RAG SYSTEM LOADED")
print("=" * 40)
print("📚 Optimized for technical documents like:")
print("   • Academic papers • Math textbooks • CS papers")
print("   • Physics texts • Engineering docs")
print()
print("🚀 Quick start: rag = setup_technical_rag()")
print("🌐 Google Colab: rag = colab_setup()")
print()
print("💡 This version provides much better answers for complex technical questions!")

🧠 TECHNICAL PDF RAG SYSTEM LOADED
📚 Optimized for technical documents like:
   • Academic papers • Math textbooks • CS papers
   • Physics texts • Engineering docs

🚀 Quick start: rag = setup_technical_rag()
🌐 Google Colab: rag = colab_setup()

💡 This version provides much better answers for complex technical questions!


In [85]:
rag = setup_technical_rag()

🧠 TECHNICAL PDF RAG SETUP
Optimized for: Math, CS, Physics, Engineering docs

🚀 Initializing Technical PDF RAG System...
Loading embeddings model (optimized for technical content)...
Loading text generation model...


Device set to use mps:0


✅ Technical RAG system ready!
📋 NEXT STEPS:
1. Upload PDF to Jupyter (drag & drop or Upload button)
2. Run: list_pdf_files()
3. Run: rag.load_pdf_from_path('filename.pdf')
4. Run: rag.create_vectorstore()
5. Ask: ask_question(rag, 'your question')


In [87]:
rag.load_pdf_from_path('GeometricDeepLearning.pdf')

📄 Loading PDF: GeometricDeepLearning.pdf
✅ Loaded 160 pages, split into 341 chunks


[Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-05-04T00:26:27+00:00', 'author': '', 'keywords': '', 'moddate': '2021-05-04T00:26:27+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'GeometricDeepLearning.pdf', 'total_pages': 160, 'page': 0, 'page_label': 'i'}, page_content='Geometric Deep Learning\nGrids, Groups, Graphs,\nGeodesics, and Gauges\nMichael M. Bronstein1, Joan Bruna2, Taco Cohen3, Petar VeliŁković4\nMay 4, 2021\n1Imperial College London / USI IDSIA / Twitter\n2New York University\n3Qualcomm AI Research. Qualcomm AI Research is an initiative of Qualcomm\nTechnologies, Inc.\n4DeepMind\narXiv:2104.13478v2  [cs.LG]  2 May 2021'),
 Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-05-04T00:26:27+00:00', 'author': '', 'keywords': '', 'moddate':

In [89]:
rag.create_vectorstore()

🔄 Creating embeddings optimized for technical content...
⏳ This may take a few minutes for technical documents...
✅ Technical vectorstore created!


In [91]:
ask_question(rag, 'Why do symmetric groups correspond to graph neural networks?')

These ideas culminated in Convolutional Neural Networks in the seminal work of Yann LeCun and co-authors (LeCun et al., 1998). The ﬁrst work to take a representation-theoretical view on invariant and equivariant neural networks was performed by Wood and Shawe-Taylor (1996), unfortunately rarely cited. More recent incarnations of these ideas include the works of Makadia et al. (2007); Esteves et al.


In [93]:
ask_question(rag,'What has Felix Klein got to do with this?')

‘geometry’hasbeen synonymous withEuclidean geometry, as no other types of geometry existed. Euclid’s monopoly came to an end in the nineteenth century, with examples of non-Euclidean geometries constructed by Lobachevesky, Bolyai, Gauss, and Riemann. Towards the end of that century, these studies had diverged into disparate ﬁelds, with mathematicians and philosophers debating the validity of and relations between these geometries as well as the nature of the “one true geometry”. A way out of this pickle was shown by a young mathematician Felix Klein, appointed in 1872 as professor in the small Bavarian University of Erlangen.


In [97]:
ask_question(rag,'What is Geometric Deep Learning?')

Xu et al. (2020b) have proposed a geometric argument for what is required of an extrapolating GNN backed by rectiﬁer activations: its components and featurisation would need to be designed so as to make its constituent modules (e.g. message function) learn onlylinear target functions. Bevilacqua et al.


## A slightly different model

In [101]:
# Improved Free PDF RAG System - Better Models & Real Text Generation
# Install required packages first:
# !pip install pypdf langchain langchain-community sentence-transformers transformers torch faiss-cpu

import os
from typing import List, Dict, Any
import re
import warnings
warnings.filterwarnings("ignore")

# Core libraries
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.schema import Document
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

class ImprovedFreePDFRag:
    def __init__(self):
        """
        Initialize RAG system with better free models
        """
        print("🚀 Initializing Improved Free PDF RAG System...")
        
        # Better embedding model for technical content
        print("📊 Loading embedding model (optimized for academic content)...")
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-mpnet-base-v2",
            model_kwargs={
                'device': 'cpu',
                'trust_remote_code': True
            }
        )
        
        # Use a much better text generation model
        print("🧠 Loading text generation model (this may take a moment)...")
        self.generator = pipeline(
            "text-generation",
            model="microsoft/DialoGPT-medium",
            tokenizer="microsoft/DialoGPT-medium",
            max_length=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=50256,
            eos_token_id=50256
        )
        
        # Also load a summarization model for better context processing
        print("📝 Loading summarization model...")
        self.summarizer = pipeline(
            "summarization",
            model="facebook/bart-large-cnn",
            max_length=150,
            min_length=30,
            do_sample=False
        )
        
        self.vectorstore = None
        self.documents = []
        self.pdf_name = None
        print("✅ Improved RAG system ready!")
    
    def load_pdf_from_path(self, pdf_path: str) -> List[Document]:
        """Load PDF with optimized processing"""
        if not os.path.exists(pdf_path):
            print(f"❌ PDF file not found: {pdf_path}")
            pdf_files = [f for f in os.listdir('.') if f.endswith('.pdf')]
            if pdf_files:
                print("📄 Available PDF files:")
                for f in pdf_files:
                    print(f"   • {f}")
            raise FileNotFoundError(f"PDF file not found: {pdf_path}")
        
        print(f"📄 Loading PDF: {pdf_path}")
        self.pdf_name = pdf_path
        
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
        
        # Optimized chunking for academic/technical content
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1200,  # Good size for technical concepts
            chunk_overlap=400,  # Substantial overlap for continuity
            length_function=len,
            separators=["\n\n", "\n", ". ", "? ", "! ", "; ", " ", ""]
        )
        
        split_docs = text_splitter.split_documents(documents)
        
        # Clean up the documents
        cleaned_docs = []
        for doc in split_docs:
            # Remove excessive whitespace and clean text
            cleaned_text = re.sub(r'\s+', ' ', doc.page_content.strip())
            if len(cleaned_text) > 100:  # Only keep substantial chunks
                doc.page_content = cleaned_text
                cleaned_docs.append(doc)
        
        self.documents = cleaned_docs
        print(f"✅ Loaded {len(documents)} pages → {len(cleaned_docs)} quality chunks")
        return cleaned_docs
    
    def create_vectorstore(self):
        """Create vectorstore"""
        if not self.documents:
            raise ValueError("No documents loaded. Call load_pdf_from_path() first.")
        
        print("🔄 Creating vectorstore with technical embeddings...")
        print("⏳ Processing embeddings (this takes ~1-3 minutes)...")
        
        self.vectorstore = FAISS.from_documents(self.documents, self.embeddings)
        print("✅ Vectorstore ready!")
    
    def query(self, question: str, k: int = 4) -> str:
        """
        Query with improved answer generation
        """
        if not self.vectorstore:
            raise ValueError("Vectorstore not created. Call create_vectorstore() first.")
        
        # Retrieve relevant documents
        docs = self.vectorstore.similarity_search(question, k=k)
        
        # Method 1: Try to find direct explanations in context
        direct_answer = self._find_direct_answer(question, docs)
        if direct_answer:
            return direct_answer
        
        # Method 2: Use summarization to create coherent answer
        summary_answer = self._create_summary_answer(question, docs)
        if summary_answer:
            return summary_answer
        
        # Method 3: Return best context with explanation
        if docs:
            best_context = docs[0].page_content
            return f"Based on the document: {best_context[:500]}{'...' if len(best_context) > 500 else ''}"
        
        return "I couldn't find relevant information to answer that question."
    
    def _find_direct_answer(self, question: str, docs: List[Document]) -> str:
        """Look for direct explanations in the retrieved documents"""
        question_words = set(word.lower() for word in question.split() if len(word) > 2)
        
        for doc in docs:
            text = doc.page_content
            sentences = [s.strip() for s in text.split('.') if len(s.strip()) > 30]
            
            # Look for sentences that contain question keywords
            relevant_sentences = []
            for sentence in sentences:
                sentence_words = set(word.lower() for word in sentence.split())
                overlap = len(question_words.intersection(sentence_words))
                
                if overlap >= 2:  # Good keyword overlap
                    relevant_sentences.append((overlap, sentence))
            
            if relevant_sentences:
                # Sort by relevance and combine top sentences
                relevant_sentences.sort(reverse=True, key=lambda x: x[0])
                top_sentences = [sent[1] for sent in relevant_sentences[:3]]
                
                # Create coherent explanation
                explanation = '. '.join(top_sentences) + '.'
                
                # Clean up formatting
                explanation = re.sub(r'\s+', ' ', explanation)
                
                if len(explanation) > 100:  # Ensure substantial answer
                    return explanation
        
        return None
    
    def _create_summary_answer(self, question: str, docs: List[Document]) -> str:
        """Create answer using summarization model"""
        try:
            # Combine relevant contexts
            combined_text = " ".join([doc.page_content for doc in docs[:2]])
            
            # Limit text length for summarizer
            if len(combined_text) > 1000:
                combined_text = combined_text[:1000]
            
            # Create context-aware prompt
            prompt_text = f"Question: {question}\n\nContext: {combined_text}\n\nBased on this context:"
            
            # Generate summary
            summary_result = self.summarizer(
                combined_text,
                max_length=100,
                min_length=20,
                do_sample=False
            )
            
            summary = summary_result[0]['summary_text']
            
            # Clean up summary
            summary = summary.strip()
            if not summary.endswith('.'):
                summary += '.'
            
            return summary
            
        except Exception as e:
            print(f"Summarization failed: {e}")
            return None
    
    def get_context(self, question: str, k: int = 3):
        """Get relevant context without generating answer - useful for debugging"""
        if not self.vectorstore:
            print("Vectorstore not created yet")
            return
        
        docs = self.vectorstore.similarity_search(question, k=k)
        
        print(f"📖 Context for: '{question}'")
        print("=" * 60)
        
        for i, doc in enumerate(docs, 1):
            page = doc.metadata.get('page', '?')
            content = doc.page_content[:300].replace('\n', ' ')
            print(f"\n{i}. Page {page}:")
            print(f"   {content}...")
        
        return docs
    
    def save_vectorstore(self, path: str = "technical_vectorstore"):
        """Save vectorstore"""
        if not self.vectorstore:
            raise ValueError("No vectorstore to save")
        self.vectorstore.save_local(path)
        print(f"💾 Saved to {path}")
    
    def load_vectorstore(self, path: str = "technical_vectorstore"):
        """Load vectorstore"""
        self.vectorstore = FAISS.load_local(path, self.embeddings, allow_dangerous_deserialization=True)
        print(f"📂 Loaded from {path}")

In [103]:
rag = setup_technical_rag()

🧠 TECHNICAL PDF RAG SETUP
Optimized for: Math, CS, Physics, Engineering docs

🚀 Initializing Technical PDF RAG System...
Loading embeddings model (optimized for technical content)...
Loading text generation model...


Device set to use mps:0


✅ Technical RAG system ready!
📋 NEXT STEPS:
1. Upload PDF to Jupyter (drag & drop or Upload button)
2. Run: list_pdf_files()
3. Run: rag.load_pdf_from_path('filename.pdf')
4. Run: rag.create_vectorstore()
5. Ask: ask_question(rag, 'your question')


In [105]:
rag.load_pdf_from_path('GeometricDeepLearning.pdf')

📄 Loading PDF: GeometricDeepLearning.pdf
✅ Loaded 160 pages, split into 341 chunks


[Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-05-04T00:26:27+00:00', 'author': '', 'keywords': '', 'moddate': '2021-05-04T00:26:27+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'GeometricDeepLearning.pdf', 'total_pages': 160, 'page': 0, 'page_label': 'i'}, page_content='Geometric Deep Learning\nGrids, Groups, Graphs,\nGeodesics, and Gauges\nMichael M. Bronstein1, Joan Bruna2, Taco Cohen3, Petar VeliŁković4\nMay 4, 2021\n1Imperial College London / USI IDSIA / Twitter\n2New York University\n3Qualcomm AI Research. Qualcomm AI Research is an initiative of Qualcomm\nTechnologies, Inc.\n4DeepMind\narXiv:2104.13478v2  [cs.LG]  2 May 2021'),
 Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-05-04T00:26:27+00:00', 'author': '', 'keywords': '', 'moddate':

In [107]:
rag.create_vectorstore()

🔄 Creating embeddings optimized for technical content...
⏳ This may take a few minutes for technical documents...
✅ Technical vectorstore created!


In [109]:
ask_question(rag,'What is Geometric Deep Learning?')

Xu et al. (2020b) have proposed a geometric argument for what is required of an extrapolating GNN backed by rectiﬁer activations: its components and featurisation would need to be designed so as to make its constituent modules (e.g. message function) learn onlylinear target functions. Bevilacqua et al.


In [113]:
ask_question(rag, "How do group actions relate to neural network architectures?")

can also act on the space ofimageson the plane (by translating, rotating and ﬂipping the grid of pixels), as well as on the representation spaces learned by a neural network. More precisely, if we have a groupG acting onΩ, we automatically obtain an action ofG on the spaceX(Ω): (g.x)(u) = x(g−1u). (3) Due to the inverse ong, this is indeed a valid group action, in that we have (g.(h.x))(u) = ((gh).x)(u).
